In [ ]:
import requests
import json

API_KEY = "ba979c51e5a6d44c250e53ab25c603bb"

url = f"https://api.openweathermap.org/data/2.5/weather?q=Islamabad&appid={API_KEY}&units=metric"

response = requests.get(url)

print("Status Code:", response.status_code)

data = response.json()

print(json.dumps(data, indent=4))


StatementMeta(, , -1, SessionError, , SessionError, True)

InvalidHttpRequest: [TooManyRequestsForCapacity] [TooManyRequestsForCapacity] HTTP Response code 430: This Spark job can’t be run because you’ve hit a Spark compute or API rate limit. To proceed, cancel an active Spark job through the Monitoring hub, choose a larger capacity SKU, or try again later. For more visibility and control, go to Workspace settings → Job management (Job Concurrency & Queue Monitoring) to review running and queued Spark jobs, understand capacity contention, and take action as needed. [Learn more at 'https://go.microsoft.com/fwlink/?linkid=2356970&clcid=0x409']. HTTP status code: 430.

In [ ]:
import requests
import json

# API Key
API_KEY = "ba979c51e5a6d44c250e53ab25c603bb"

# API URL
url = f"https://api.openweathermap.org/data/2.5/weather?q=Islamabad&appid={API_KEY}&units=metric"

# Fetch Data
response = requests.get(url)

# Convert response to JSON
data = response.json()

# Convert JSON dictionary to string
json_data = json.dumps(data)

# Create Spark DataFrame
df = spark.read.json(sc.parallelize([json_data]))

# Show DataFrame
df.show()

# Save Bronze Layer
df.write.format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true")\
    .saveAsTable("bronze_weather")

StatementMeta(, , -1, Cancelled, , Cancelled, True)

In [ ]:
from pyspark.sql.functions import col

# Read Bronze Layer
df = spark.table("bronze_weather")

# Flatten & Clean Data
silver_df = df.select(
    col("name").alias("city"),
    col("main.temp").alias("temperature"),
    col("main.humidity").alias("humidity"),
    col("weather")[0]["main"].alias("weather_condition"),
    col("wind.speed").alias("wind_speed"),
    col("sys.country").alias("country")
)

# Show Silver Data
silver_df.show()

# Save Silver Layer
silver_df.write.format("delta") \
    .mode("overwrite") \
    .saveAsTable("silver_weather")

StatementMeta(, , -1, Cancelled, , Cancelled, True)

In [ ]:
from pyspark.sql.functions import when, col

# Read Silver Layer
df = spark.table("silver_weather")

# Create Gold Layer
gold_df = df.withColumn(
    "temp_category",
    when(col("temperature") < 10, "Cold")
    .when(col("temperature") < 20, "Moderate")
    .otherwise("Hot")
)

# Show Gold Data
gold_df.show()

# Save Gold Layer
gold_df.write.format("delta") \
    .mode("overwrite") \
    .saveAsTable("gold_weather")

StatementMeta(, , -1, Cancelled, , Cancelled, True)

In [ ]:
import requests
import json
from pyspark.sql.functions import current_timestamp

# API Key
API_KEY = "ba979c51e5a6d44c250e53ab25c603bb"

# API URL
url = f"https://api.openweathermap.org/data/2.5/weather?q=Islamabad&appid={API_KEY}&units=metric"

# Fetch API data
response = requests.get(url)

# Convert to JSON
data = response.json()

# Create Spark DataFrame
json_data = json.dumps(data)

df = spark.read.json(sc.parallelize([json_data]))

# Add ingestion timestamp
df = df.withColumn(
    "ingestion_time",
    current_timestamp()
)

# Show result
df.show()

df.write.format("delta") \
    .mode("append") \
    .option("mergeSchema", "true")\
    .saveAsTable("bronze_weather_history")

StatementMeta(, , -1, Cancelled, , Cancelled, True)

In [ ]:
from pyspark.sql.functions import col

# Read Bronze History
df = spark.table("bronze_weather_history")

# Flatten JSON + retain timestamp
silver_df = df.select(
    col("name").alias("city"),
    col("main.temp").alias("temperature"),
    col("main.humidity").alias("humidity"),
    col("weather")[0]["main"].alias("weather_condition"),
    col("wind.speed").alias("wind_speed"),
    col("sys.country").alias("country"),
    col("ingestion_time")
)

# Show data
silver_df.show()

# Save Silver history
silver_df.write.format("delta") \
    .mode("overwrite") \
    .saveAsTable("silver_weather_history")

StatementMeta(, , -1, Cancelled, , Cancelled, True)

In [ ]:
from pyspark.sql.functions import when, col

# Read Silver History
df = spark.table("silver_weather_history")

# Create Gold Layer
gold_df = df.withColumn(
    "temp_category",
    when(col("temperature") < 10,"Cold")
    .when(col("temperature") < 20,"Moderate")
    .otherwise("Hot")
)

# Show data
gold_df.show()

# Save Gold History
gold_df.write.format("delta") \
    .mode("overwrite") \
    .saveAsTable("gold_weather_history")

StatementMeta(, , -1, Cancelled, , Cancelled, True)

In [ ]:
from pyspark.sql.functions import year, month, col

# Read Gold History
df = spark.table("gold_weather_history")

# Create partition columns
partitioned_df = df.withColumn(
    "year",
    year(col("ingestion_time"))
).withColumn(
    "month",
    month(col("ingestion_time"))
)

# Show data
partitioned_df.show()

# Save partitioned table
partitioned_df.write.format("delta") \
    .mode("overwrite") \
    .partitionBy("year","month") \
    .saveAsTable("gold_weather_partitioned")

StatementMeta(, , -1, Cancelled, , Cancelled, True)

In [ ]:
spark.sql("SHOW TABLES").show(truncate=False)

StatementMeta(, , -1, Cancelled, , Cancelled, True)

In [ ]:
spark.sql("SELECT * FROM gold_weather_partitioned").show()

StatementMeta(, , -1, Cancelled, , Cancelled, True)

In [ ]:
df = spark.table("gold_weather_partitioned")

df.write.format("delta") \
    .mode("overwrite") \
    .saveAsTable("weather_dashboard")

StatementMeta(, , -1, Cancelled, , Cancelled, True)